In [1]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import aneurysm_3Dsegmentation_in_CTA.model_utility.tuning as tuner
import segmentation_models_3D as sm3
import keras_tuner as kt
import os
import shutil

I0000 00:00:1793780000.832629   13406 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Segmentation Models: using `tf.keras` framework.


In [2]:
# loading data tensors
dataSource = "data/CTA nii" 

sizeContainer = tts.dataSplitPerSize(dataSource)

trainSet , testSet  , _ = tts.dataSplitPerSample(
    [
        sizeContainer["S"] ,
        sizeContainer["M"] ,
        sizeContainer["L"]
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet , mergeTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet  , mergeTestSet  = tts.dataTensorLoading(testSet)


In [ ]:
# check for image pairs

for name in zip(imgTrainSet , labelTrainSet , mergeTrainSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print("============")

print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet , mergeTestSet , refTestSet) :
    print(name[0])
    print(name[1])
    print(name[2])
    print(name[3])
    print("============")

In [ ]:
# pipeline configuring

geo      = utl.randomGeo(p=0.5)
crop     = utl.volume_crop((64 , 64 , 64) , paddDim=80)
tile     = utl.tile(
    tile_dim=[1 , 1 , 1 , 1 , 3]
)

setShape = utl.setShape(
    imgShape=[None , 64 , 64 , 64 , 3] ,
    labelShape=[None , 64 , 64 , 64 , 1]
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64 , tf.float64])
def rimg(imgPath , labelPath , mergePath) :
    return utl.read_img(imgPath , labelPath , mergePath)
def read_img(img , label , merge) :
    imglbl = rimg(img , label , merge)
    img   = imglbl[0]
    label = imglbl[1]
    merge = imglbl[2]
    return img , label , merge

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label

# windowing configuration part
normalizer   = utl.fixedRangeNormalization(min_val=0 , max_val=3071 , clip=True)
fixwindower   = window.windowing(ww=395 , wl=274)

In [ ]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet , mergeTrainSet))
    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )

    .cache("cache64/myCacheTrain")
    .shuffle(buffer_size=20 , seed=42 , reshuffle_each_iteration=True)
    .map(
        rot ,
        num_parallel_calls=2
    )
    .map(
        geo.flipX ,
        num_parallel_calls=2
    )
    .map(
        geo.flipY ,
        num_parallel_calls=2
    )
    .map(
        geo.flipZ ,
        num_parallel_calls=2
    )
    .map(
        utl.channelize ,
        num_parallel_calls=4
    )
    .batch(batch_size=4)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        fixwindower.apply ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTestSet , labelTestSet , mergeTestSet)) 

    .map(
        read_img , 
        num_parallel_calls=2
    )
    .map(
        crop.cropping ,
        num_parallel_calls=2
    )
    .map( 
        utl.channelize ,
        num_parallel_calls=2
    )
    .cache("cache64/myCacheValid") 
    .batch(batch_size=2)
    .map(
        utl.convert_to_channel_last ,
        num_parallel_calls=4
    )
    .map(
        tile.tile ,
        num_parallel_calls=4
    )
    .map(
        fixwindower.apply ,
        num_parallel_calls=4
    )
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
    .map(
        utl.cast32 ,
        num_parallel_calls=4
    )
    .map(
        setShape.set , 
        num_parallel_calls=4
    )
)

technical run

In [ ]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=[1.3 , 0.2])

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(64 , 64 , 64 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features="default"
)


# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze_Conv2_unit1ToEnd_res18 , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)

'''
# add auto windowing layer

autoWindwLayer = window.autoWindowing(
    initialLevel  = 274 ,
    initialWindow = 395
)

newInput = keras.Input(shape=(32 , 32 , 32 , 3))
x = autoWindwLayer(newInput)
x = model(x)
model = keras.Model(
    inputs=newInput ,
    outputs=x
)
'''

'''
# add auto windowing layer (multi channel)

autoWindwLayer = window.autoWindowingMultiChannel(
    initialLevel  = [233 , 274 , 322] ,
    initialWindow = [395 , 395 , 395]
)

newInput = keras.Input(shape=(32 , 32 , 32 , 1))
x = autoWindwLayer(newInput)
x = model(x)
model = keras.Model(
    inputs=newInput ,
    outputs=x
)
'''

# compilation
lr = keras.optimizers.schedules.PiecewiseConstantDecay(
    [
        13026 ,
        26052 ,

    ] ,
    [
        5e-4 ,
        1e-4 ,
        1e-5
    ]
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice , 
    ] ,
)
# callbacks
bestModel = keras.callbacks.ModelCheckpoint(
    filepath = "" , 
    monitor  = "val_loss" , 
    mode     = "min" ,
    save_best_only    = True ,
    save_weights_only = True
) 
logger    = keras.callbacks.CSVLogger("")

# model training
history = model.fit(
    x = dataloaderTrain ,
    epochs=500 ,
    validation_data = dataloaderValid ,
    callbacks = [
        logger , 
        bestModel
    ]
)

# Save Model After Training (last epoch)
keras.saving.save_weights(
    model=model ,
    filepath = "" ,
    overwrite=True
)

Unet HyperModel

In [ ]:
# building hyperModel
hp = kt.HyperParameters()


res18HyperModel = tuner.HyperUnetResNet(
    backbone="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features="default" ,
    steps=[
        5530 ,
        11060
    ]
)

res18HyperModel.set_tuning_param(
    unfreeze_point = [
        conf.unfreeze_Conv2_unit1ToEnd_res18 ,
        conf.unfreeze_Conv3_unit1ToEnd_res18 ,
        conf.unfreeze_Conv4_unit1ToEnd_res18  
    ] ,
    alphForFocalLoss=[0.25 , 0.35] ,
    alphaForWeightedFocalDiceloss=[
        [1.3] , 
        [0.2] , 
    ] ,

    learning_rate=[   
        [5e-4] ,
        [1e-4] ,
        [1e-5]
    ]

)

In [ ]:
# tuning model
tuner = kt.GridSearch(
    res18HyperModel ,
    objective="val_loss" ,
    directory="tuning_logs" ,
    project_name="hypermodel_D1"
)

tuner.search(
    x = dataloaderTrain ,
    epochs=200 ,
    validation_data = dataloaderValid ,
)

In [ ]:
best_confguration = tuner.get_best_hyperparameters()[0].get_config()
pprint(best_confguration["values"])